<a href="https://colab.research.google.com/github/abhijadhav14/AES-Encryption/blob/main/Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Linear Regression using PySpark MLlib

# Import Libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import random

# Initialize Spark Session
spark = SparkSession.builder.appName("LinearRegressionExample").getOrCreate()

#Dataset
num_samples = 1000
data = []
for _ in range(num_samples):
    sqft = random.randint(800, 3000)
    bedrooms = random.randint(1, 5)

    price = (sqft * 200 + bedrooms * 30000) + random.uniform(-50000, 50000)
    data.append((sqft, bedrooms, price))

columns = ["sqft", "bedrooms", "price"]
df = spark.createDataFrame(data, columns)

# Combine Features into a Single Vector (Required for MLlib)
assembler = VectorAssembler(inputCols=["sqft", "bedrooms"], outputCol="features")
df_assembled = assembler.transform(df)

# Select Features and Target
final_df = df_assembled.select("features", "price")

# Split Data
train_data, test_data = final_df.randomSplit([0.8, 0.2], seed=42)

# Create and Train Linear Regression Model
lr = LinearRegression(featuresCol="features", labelCol="price", maxIter=10)
lr_model = lr.fit(train_data)

# Make Predictions
predictions = lr_model.transform(test_data)

# Evaluate Model (Using Root Mean Squared Error - RMSE)
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print("Root Mean Squared Error (RMSE):", rmse)

# View Example Predictions
print("\nPredictions for Test Data:")
predictions.select("features", "price", "prediction").show(truncate=False)


Root Mean Squared Error (RMSE): 29536.30978244593

Predictions for Test Data:
+------------+------------------+------------------+
|features    |price             |prediction        |
+------------+------------------+------------------+
|[801.0,5.0] |323475.9802904717 |305659.6787494837 |
|[805.0,4.0] |293187.72870520037|277568.5306638691 |
|[824.0,5.0] |351395.9744385518 |310331.3400349511 |
|[856.0,1.0] |180436.05748260245|201216.59902151345|
|[893.0,3.0] |220083.22151765783|266539.10205561365|
|[903.0,5.0] |283142.8055188174 |326377.4809719913 |
|[920.0,3.0] |320831.0965827319 |272023.22617333627|
|[931.0,5.0] |325690.9320799206 |332064.72079777776|
|[980.0,3.0] |267554.8845742142 |284210.1686571643 |
|[985.0,3.0] |263240.98598973843|285225.7471974833 |
|[997.0,2.0] |220713.645655965  |258759.5247763791 |
|[1013.0,5.0]|324986.7804240237 |348720.2088590093 |
|[1019.0,3.0]|276693.1672858015 |292131.6812716525 |
|[1032.0,2.0]|248920.00000435187|265868.5745586121 |
|[1070.0,2.0]|285677.